In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
import pickle
import os

# Load data
data_path = os.path.join('..', 'data', 'apriori_market_basket_10000.csv')
df = pd.read_csv(data_path)

# Preprocess: pivot data to transaction/item format
basket = (df.groupby(['transaction_id', 'item'])['quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('transaction_id'))

# Convert quantities to 0/1 (binary) for Apriori
def encode_item_value(x):
    if x <= 0:
        return False
    if x >= 1:
        return True
basket_sets = basket.map(encode_item_value)

# Train Apriori
frequent_itemsets = apriori(basket_sets, min_support=0.05, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

# Save model
model_data = {
    'rules': rules,
    'frequent_itemsets': frequent_itemsets,
    'basket_columns': basket.columns.tolist()
}

with open('model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

# Save metrics for the app
metrics = {
    'avg_support': rules['support'].mean(),
    'avg_confidence': rules['confidence'].mean(),
    'avg_lift': rules['lift'].mean(),
    'num_rules': len(rules)
}
with open('metrics.pkl', 'wb') as f:
    pickle.dump(metrics, f)

print("Apriori training completed successfully.")
print(f"Rules found: {len(rules)}")


Apriori training completed successfully.
Rules found: 60
